# Inference

Inference on my SFT model on raw data

In [4]:
import os
os.chdir('/vol/bitbucket/lc3223/tmux-test/')
!pwd

/vol/bitbucket/lc3223/tmux-test


In [5]:
import torch

if torch.cuda.is_available():
    device = torch.cuda.current_device()
    gpu_name = torch.cuda.get_device_name(device)
    
    total_memory = torch.cuda.get_device_properties(device).total_memory
    allocated_memory = torch.cuda.memory_allocated(device)
    reserved_memory = torch.cuda.memory_reserved(device)
    free_memory = total_memory - reserved_memory
    
    print(f"GPU: {gpu_name}")
    print(f"Total Memory:     {total_memory / 1e6:.2f} MB | {total_memory / 1e9:.2f} GB")
    print(f"Allocated Memory: {allocated_memory / 1e6:.2f} MB | {allocated_memory / 1e9:.2f} GB")
    print(f"Reserved Memory:  {reserved_memory / 1e6:.2f} MB | {reserved_memory / 1e9:.2f} GB")
    print(f"Free Memory:      {free_memory / 1e6:.2f} MB | {free_memory / 1e9:.2f} GB")
else:
    print("No CUDA GPU available")

GPU: NVIDIA A30
Total Memory:     25338.05 MB | 25.34 GB
Allocated Memory: 0.00 MB | 0.00 GB
Reserved Memory:  0.00 MB | 0.00 GB
Free Memory:      25338.05 MB | 25.34 GB


In [7]:
from transformers import AutoModelForCausalLM, AutoTokenizer, TextStreamer

BASE_MODEL_NAME = "Qwen/Qwen3.5-2B"
MODEL_NAME = "checkpoint_models"

tok = AutoTokenizer.from_pretrained(MODEL_NAME, device_map="auto")
model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, device_map="auto")
base_model = AutoModelForCausalLM.from_pretrained(BASE_MODEL_NAME, device_map="auto")

The fast path is not available because one of the required library is not installed. Falling back to torch implementation. To install follow https://github.com/fla-org/flash-linear-attention#installation and https://github.com/Dao-AILab/causal-conv1d
Current model requires 256 bytes of buffer for offloaded layers, which seems does not fit any GPU's remaining memory. If you are experiencing a OOM later, please consider using offload_buffers=True.


Loading weights:   0%|          | 0/320 [00:00<?, ?it/s]

Current model requires 256 bytes of buffer for offloaded layers, which seems does not fit any GPU's remaining memory. If you are experiencing a OOM later, please consider using offload_buffers=True.


Loading weights:   0%|          | 0/320 [00:00<?, ?it/s]

In [8]:
print(f"[INFO] Model on device: {model.device}")
print(f"[INFO] Model using dtype: {model.dtype}")

[INFO] Model on device: cpu
[INFO] Model using dtype: torch.bfloat16


In [23]:
messages = [
    {"role": "user", "content": "我在看要不要做Eurostar去阿姆斯特丹\n我算了之後\n更便宜\n也比較方便\n只是要做四小時"},
]

tokenized_chat = tok.apply_chat_template(messages, tokenize=False, add_generation_prompt=True, return_tensors="pt")
tokenized_chat

'<|im_start|>user\n我在看要不要做Eurostar去阿姆斯特丹\n我算了之後\n更便宜\n也比較方便\n只是要做四小時<|im_end|>\n<|im_start|>assistant\n<think>\n\n</think>\n\n'

In [24]:

inputs = tok(tokenized_chat, return_tensors="pt")
streamer = TextStreamer(tok)

print("[INFO] Fine-tuned model output:")
_ = model.generate(**inputs, streamer=streamer)

print("\n[INFO] Base model output:")
_ = base_model.generate(**inputs, streamer=streamer)

[INFO] Fine-tuned model output:
<|im_start|>user
我在看要不要做Eurostar去阿姆斯特丹
我算了之後
更便宜
也比較方便
只是要做四小時<|im_end|>
<|im_start|>assistant
<think>

</think>

對呀
那就在看看！<|im_end|>


Setting `pad_token_id` to `eos_token_id`:248044 for open-end generation.



[INFO] Base model output:
<|im_start|>user
我在看要不要做Eurostar去阿姆斯特丹
我算了之後
更便宜
也比較方便
只是要做四小時<|im_end|>
<|im_start|>assistant
<think>

</think>

你的計算非常準確！從倫敦出發，**Eurostar 去阿姆斯特丹**確實是目前最
